In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

/Users/choyoungrae/Desktop/book-study/books/RAG 마스터 랭체인으로 완성하는 LLM 서비스/rag-master-practice-260326/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
from langchain_core.documents import Document

In [7]:
url = "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch02.%20RAG/Data/2024_KB_%EB%B6%80%EB%8F%99%EC%82%B0_%EB%B3%B4%EA%B3%A0%EC%84%9C_%EC%B5%9C%EC%A2%85.pdf"
loader = PyPDFLoader(url)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print("분할된 청크의 수:", len(chunks))

분할된 청크의 수: 135


In [10]:
embedding_function = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embedding_function)

In [11]:
print("문서의 수:", vector_store._collection.count())

문서의 수: 405


In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [15]:
template = """
당신은 KB 부동산 보고서 전문가입니다.
다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

[컨텍스트]
{context}
""".strip()

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("placeholder", "{chat_history}"),
    ("human", "{question}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [16]:
print(prompt.format(context="컨텍스트 예시", chat_history=["대화 기록 예시1", "대화 기록 예시2"], question="질문 예시"))

System: 당신은 KB 부동산 보고서 전문가입니다.
다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

[컨텍스트]
컨텍스트 예시
Human: 대화 기록 예시1
Human: 대화 기록 예시2
Human: 질문 예시


In [21]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

In [22]:
chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"])),
    )
    | prompt
    | model
    | StrOutputParser()
)

In [23]:
chat_history = ChatMessageHistory()
chain_with_memory = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [25]:
def chat_with_bot():
    session_id = "user_session"
    print("KB 부동산 보고서 챗봇입니다. 질문해 주세요. (종료하려면 'quit' 입력)")
    while True:
        user_input = input("사용자: ")
        if user_input.lower() == 'quit':
            break

        response = chain_with_memory.invoke(
            {"question": user_input},
            {"configurable": {"session_id": session_id}}
        )

        print("챗봇:", response)

chat_with_bot()

KB 부동산 보고서 챗봇입니다. 질문해 주세요. (종료하려면 'quit' 입력)
챗봇: 2024년 주택시장에 대한 전망은 여러 요인에 따라 달라질 수 있습니다. 수도권과 기타 지방의 입주물량과 전세가격 변동률을 고려할 때, 수도권의 경우 입주물량이 증가하면서 전세가격이 안정세를 보일 가능성이 있습니다. 반면, 기타 지방은 입주물량과 전세가격의 변동이 다를 수 있어 지역별로 차별화된 시장 상황이 나타날 수 있습니다.

전반적으로, 주택시장은 공급과 수요의 균형, 금리 변화, 경제 상황 등에 따라 영향을 받을 것이므로, 이러한 요소들을 종합적으로 고려해야 할 것입니다. 추가적인 세부사항이나 특정 지역에 대한 정보가 필요하시면 말씀해 주세요.
